# INE Mortality Replication Walkthrough

This notebook is a readable companion to `ine_methodology_replication.py`.

It keeps the implementation in one Python script, but exposes the workflow step by step:

1. load the module
2. validate required inputs
3. parse observed and benchmark data
4. run the replication pipeline
5. inspect the main outputs
6. write the Excel outputs and validation workbook

The methodology follows the public INE documentation for the `2024-2073` projected mortality tables.

## How to use this notebook

- The notebook is meant for reading and experimentation.
- The script `ine_methodology_replication.py` remains the implementation source of truth.
- If you want to regenerate all Excel outputs exactly as in the repo workflow, execute all cells in order.

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "ine_methodology_replication.py").exists():
    script_path = cwd / "ine_methodology_replication.py"
elif (cwd / "code" / "replication" / "ine_methodology_replication.py").exists():
    script_path = cwd / "code" / "replication" / "ine_methodology_replication.py"
else:
    raise FileNotFoundError("Could not locate ine_methodology_replication.py from the current working directory.")

spec = importlib.util.spec_from_file_location("ine_methodology_replication", script_path)
imr = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(imr)

print(f"Loaded module from: {script_path}")
print(f"Project root: {imr.PROJECT_ROOT}")

In [ ]:
# Check that the required INE inputs and benchmark workbooks are present.
imr.validate_required_files()
imr.ensure_dirs()
print("All required files were found and output directories are ready.")

## Parse observed and benchmark data

The script reads:

- observed mortality-table information from INE table `27153`
- projected benchmark `qx` from INE table `36774`
- projected benchmark `e0` from INE table `36775`

In [ ]:
observed_qx, observed_ax, observed_e0, benchmark_qx, benchmark_e0 = imr.load_all_inputs()

print(f"Observed qx cells: {len(observed_qx):,}")
print(f"Observed ax cells: {len(observed_ax):,}")
print(f"Observed e0 values: {len(observed_e0):,}")
print(f"Benchmark male qx cells: {len(benchmark_qx['male']):,}")
print(f"Benchmark female qx cells: {len(benchmark_qx['female']):,}")
print(f"Benchmark male e0 years: {len(benchmark_e0['male']):,}")
print(f"Benchmark female e0 years: {len(benchmark_e0['female']):,}")

## Run the replication pipeline

This step:

- fits the bounded logit `e0` path
- interpolates the `2073` horizon profile from the model life tables
- builds projected annual `qx` and `ax` tables
- produces the baseline result and the optional `male_high_age_adjusted` variant

In [ ]:
results = imr.run_replication_pipeline(observed_qx, observed_ax, observed_e0)
list(results.keys())

In [ ]:
baseline_male = results[imr.VARIANT_BASELINE]["male"]
baseline_female = results[imr.VARIANT_BASELINE]["female"]

pd.DataFrame(
    {
        "male_e0_life_table": baseline_male.life_projection.set_index("Year")["e0_life_table"],
        "female_e0_life_table": baseline_female.life_projection.set_index("Year")["e0_life_table"],
    }
).head()

In [ ]:
summary_rows = []
for variant_name, variant_results in results.items():
    for sex in ["male", "female"]:
        result = variant_results[sex]
        generated_qx = imr.projected_qx_lookup(result.qx_projected)
        _, generated_e0_life = imr.projected_e0_lookup(result.e0_projection, result.life_projection)

        _, qx_summary = imr.compare_dicts(generated_qx, benchmark_qx[sex])
        _, e0_summary = imr.compare_dicts(generated_e0_life, benchmark_e0[sex])

        summary_rows.append(
            {
                "variant": variant_name,
                "sex": sex,
                "qx_mae_per_thousand": qx_summary.mean_abs_error,
                "e0_mae_years": e0_summary.mean_abs_error,
            }
        )

pd.DataFrame(summary_rows)

## Write the Excel outputs

These two calls reproduce the normal script behavior:

- write projected tables to `output/mortality_projection/...`
- write the validation workbook comparing the replication against INE

In [ ]:
imr.write_projection_outputs(results)
imr.write_validation_workbook(results, benchmark_qx, benchmark_e0, observed_qx)

print(f"Final output folder: {imr.FINAL_DIR}")
print(f"Validation workbook: {imr.VALIDATION_FILE}")

## Next step

Once the INE replication is considered stable, the next stage is to extend the horizon to `2100` while keeping the replication baseline separate from the longer-horizon UN assumptions.